# RQ4: Feature Importance & SHAP — Drivers of Customer Satisfaction

**Research Question:** What campaign, product, and subscription features most strongly drive customer satisfaction post-refund, and do global feature importance rankings align with local SHAP-based explanations for individual high- and low-satisfaction predictions?

**Task:** Regression + Explainability (XAI)  
**Target:** `Customer_Satisfaction_Post_Refund` (1–5 scale)  
**Models:** Random Forest, XGBoost, Gradient Boosting + SHAP TreeExplainer  
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [68]:
!pip install shap --quiet

In [69]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
sns.set_palette(PALETTE)
RANDOM_STATE = 42
OUTPUT_DIR = '/kaggle/working/'

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved figure: {path}')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'Saved table:  {path}')

print('Imports OK')

Imports OK


## 1. Data Loading & Feature Engineering

In [70]:
input_dir = '/kaggle/input'
data_files = [
    os.path.join(root, f)
    for root, dirs, files in os.walk(input_dir)
    for f in files if f.endswith('.xlsx') or f.endswith('.xls') or f.endswith('.csv')
]
print('Found files:', data_files)
FILE_PATH = data_files[0]

df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith('.csv') else pd.read_excel(FILE_PATH)
print(f'Shape: {df.shape}')

df['Has_Flash_Sale'] = df['Flash_Sale_ID'].notna().astype(int)
df['Has_Bundle']     = df['Bundle_ID'].notna().astype(int)
df['Conversion_Rate'] = np.where(df['Clicks'] > 0, df['Conversions'] / df['Clicks'], 0)

NUMERIC_FEATURES = [
    'Budget', 'Clicks', 'Conversions', 'Revenue_Generated', 'ROI',
    'Discount_Level', 'Units_Sold', 'Bundle_Price', 'Subscription_Length',
    'Has_Flash_Sale', 'Has_Bundle', 'Conversion_Rate'
]
CAT_FEATURES = ['Subscription_Tier']

X = df[NUMERIC_FEATURES + CAT_FEATURES].copy()
y = df['Customer_Satisfaction_Post_Refund'].copy()

print(f'Target range: {y.min()} – {y.max()}  |  Mean: {y.mean():.2f}  |  Std: {y.std():.2f}')

Found files: ['/kaggle/input/datasets/vanishjr/marketing-product-performance/marketing_and_product_performance.csv']
Shape: (10000, 17)
Target range: 1 – 4  |  Mean: 2.50  |  Std: 1.11


## 2. Preprocessing & Train/Test Split

In [71]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (8000, 13) | Test: (2000, 13)


## 3. Model Training

In [72]:
models = {
    'Random Forest':      RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':            XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                        random_state=RANDOM_STATE, verbosity=0),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                     max_depth=4, random_state=RANDOM_STATE)
}

fitted_models = {}
for name, reg in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', reg)])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    r2   = r2_score(y_test, pipe.predict(X_test))
    rmse = np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))
    print(f'{name}: R²={r2:.4f}  RMSE={rmse:.4f}')

Random Forest: R²=-0.0111  RMSE=1.1152
XGBoost: R²=-0.0713  RMSE=1.1479
Gradient Boosting: R²=-0.0198  RMSE=1.1200


## 4. Global Feature Importance

In [73]:
# Get feature names after transformation
dummy_pipe = fitted_models['Random Forest']
dummy_pipe.named_steps['preprocessor'].fit(X_train)
cat_feature_names = list(
    dummy_pipe.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(CAT_FEATURES)
)
ALL_FEATURE_NAMES = NUMERIC_FEATURES + cat_feature_names

importance_records = []
for name, pipe in fitted_models.items():
    importances = pipe.named_steps['model'].feature_importances_
    for feat, imp in zip(ALL_FEATURE_NAMES, importances):
        importance_records.append({'Model': name, 'Feature': feat, 'Importance': round(float(imp), 6)})

imp_df = pd.DataFrame(importance_records)
imp_pivot = imp_df.pivot(index='Feature', columns='Model', values='Importance').reset_index()
imp_pivot.columns.name = None

# Compute mean rank across models
for col in ['Random Forest', 'XGBoost', 'Gradient Boosting']:
    imp_pivot[f'{col}_Rank'] = imp_pivot[col].rank(ascending=False).astype(int)
imp_pivot['Mean_Rank'] = imp_pivot[['Random Forest_Rank','XGBoost_Rank','Gradient Boosting_Rank']].mean(axis=1).round(2)
imp_pivot = imp_pivot.sort_values('Mean_Rank')

save_table(imp_pivot, 'rq4_feature_importance_comparison.csv')
imp_pivot.head(15)

Saved table:  /kaggle/working/rq4_feature_importance_comparison.csv


,Feature,Gradient Boosting,Random Forest,XGBoost,Random Forest_Rank,XGBoost_Rank,Gradient Boosting_Rank,Mean_Rank
1,Bundle_Price,0.146503,0.122891,0.086669,1,5,1,2.33
8,ROI,0.108272,0.103565,0.087886,4,4,5,4.33
9,Revenue_Generated,0.110096,0.111302,0.082618,3,8,4,5.00
3,Conversion_Rate,0.092739,0.087637,0.094982,8,1,6,5.00
0,Budget,0.132048,0.112773,0.067201,2,12,2,5.33
13,Units_Sold,0.077764,0.095112,0.089683,7,2,9,6.00
2,Clicks,0.115291,0.096197,0.072804,6,11,3,6.67
4,Conversions,0.083239,0.097317,0.080301,5,10,7,7.33
10,Subscription_Length,0.037598,0.070190,0.088115,10,3,10,7.67
5,Discount_Level,0.078354,0.085786,0.083531,9,7,8,8.00


In [74]:
# Spearman rank correlation between models
rank_cols = {'RF': imp_pivot['Random Forest_Rank'], 'XGB': imp_pivot['XGBoost_Rank'], 'GB': imp_pivot['Gradient Boosting_Rank']}
pairs = [('RF','XGB'), ('RF','GB'), ('XGB','GB')]
corr_rows = []
for a, b in pairs:
    rho, p = spearmanr(rank_cols[a], rank_cols[b])
    corr_rows.append({'Model_A': a, 'Model_B': b, 'Spearman_rho': round(rho, 4), 'p_value': round(p, 4)})
corr_df = pd.DataFrame(corr_rows)
save_table(corr_df, 'rq4_importance_rank_correlation.csv')
print(corr_df.to_string(index=False))

Saved table:  /kaggle/working/rq4_importance_rank_correlation.csv
Model_A Model_B  Spearman_rho  p_value
     RF     XGB        0.2115   0.4680
     RF      GB        0.9471   0.0000
    XGB      GB        0.1850   0.5266


In [75]:
# Grouped bar chart — top 10 features
top10 = imp_pivot.head(10)[['Feature','Random Forest','XGBoost','Gradient Boosting']]
top10_melt = top10.melt(id_vars='Feature', var_name='Model', value_name='Importance')

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=top10_melt, x='Feature', y='Importance', hue='Model',
            ax=ax, palette=PALETTE[:3], edgecolor='white')
ax.set_title('Top-10 Feature Importances by Model (RQ4)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Feature Importance')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
ax.legend(title='Model', fontsize=9)
plt.tight_layout()
save_figure(fig, 'rq4_feature_importance_grouped_bar.pdf')
plt.show()

Saved figure: /kaggle/working/rq4_feature_importance_grouped_bar.pdf


## 5. SHAP Analysis (XGBoost)

In [76]:
xgb_pipe = fitted_models['XGBoost']
X_test_transformed = xgb_pipe.named_steps['preprocessor'].transform(X_test)
xgb_model = xgb_pipe.named_steps['model']

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test_transformed)
shap_values.feature_names = ALL_FEATURE_NAMES

print('SHAP values computed. Shape:', shap_values.values.shape)

SHAP values computed. Shape: (2000, 14)


In [77]:
# SHAP Beeswarm Summary Plot
fig = plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title('SHAP Beeswarm — Customer Satisfaction Drivers (RQ4)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq4_shap_beeswarm.pdf')
plt.show()

Saved figure: /kaggle/working/rq4_shap_beeswarm.pdf


In [78]:
# SHAP Waterfall for highest- and lowest-satisfaction predictions
y_pred_test = xgb_pipe.predict(X_test)
idx_high = int(np.argmax(y_pred_test))
idx_low  = int(np.argmin(y_pred_test))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plt.sca(axes[0])
shap.plots.waterfall(shap_values[idx_high], max_display=12, show=False)
axes[0].set_title(f'High Satisfaction (pred={y_pred_test[idx_high]:.2f})', fontsize=11, fontweight='bold')

plt.sca(axes[1])
shap.plots.waterfall(shap_values[idx_low], max_display=12, show=False)
axes[1].set_title(f'Low Satisfaction (pred={y_pred_test[idx_low]:.2f})', fontsize=11, fontweight='bold')

fig.suptitle('SHAP Waterfall — High vs Low Satisfaction (RQ4)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq4_shap_waterfall_high_low_satisfaction.pdf')
plt.show()

Saved figure: /kaggle/working/rq4_shap_waterfall_high_low_satisfaction.pdf


## 6. Conclusions

In [79]:
print('=' * 60)
print('RQ4 CONCLUSION')
print('=' * 60)
print('Top features driving Customer_Satisfaction_Post_Refund:')
for i, row in imp_pivot.head(5).iterrows():
    print(f"  #{int(row['Mean_Rank'])} {row['Feature']}")
print()
print('Outputs saved:')
for f in ['rq4_shap_beeswarm.pdf','rq4_shap_waterfall_high_low_satisfaction.pdf',
          'rq4_feature_importance_grouped_bar.pdf',
          'rq4_feature_importance_comparison.csv','rq4_importance_rank_correlation.csv']:
    print(f'  {f}')

RQ4 CONCLUSION
Top features driving Customer_Satisfaction_Post_Refund:
  #2 Bundle_Price
  #4 ROI
  #5 Revenue_Generated
  #5 Conversion_Rate
  #5 Budget

Outputs saved:
  rq4_shap_beeswarm.pdf
  rq4_shap_waterfall_high_low_satisfaction.pdf
  rq4_feature_importance_grouped_bar.pdf
  rq4_feature_importance_comparison.csv
  rq4_importance_rank_correlation.csv
